![](/Workspace/Users/sunnygupta2508@gmail.com/Databricks-Certified-Data-Engineer-Pro/Includes/images/books.png)

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
def type2_upsert(microBatchDF, batchId):
    microBatchDF.createOrReplaceTempView("updates")

    sql_query = """
        merge into books_silver a 
        using (
                select md5(book_id) as merge_key, updates.*
                from updates
                
                union all

                select null as merge_key, updates.*
                from updates
                join books_silver on books_silver.book_id = updates.book_id  
                where books_silver.current = true and books_silver.price <> updates.price

            ) b
            on a.merge_key = b.merge_key and a.current = true
            when matched  and a.price <> b.price 
            then update set a.current = false, a.end_date = b.updated
            when not matched then 
            insert (merge_key, book_id, title, author, price, current, effective_date, end_date)
            values (md5(b.book_id), b.book_id, b.title, b.author, b.price, true, b.updated, null)

    """
    microBatchDF.sparkSession.sql(sql_query)
    

In [0]:
%sql
CREATE TABLE IF NOT EXISTS books_silver
(merge_key string, book_id STRING, title STRING, author STRING, price DOUBLE, current BOOLEAN, effective_date TIMESTAMP, end_date TIMESTAMP)

In [0]:
def process_books():
    schema = "book_id STRING, title STRING, author STRING, price DOUBLE, updated TIMESTAMP"

    query = (

        spark.readStream
                .table("bronze")
                .filter("topic = 'books'")
                .select(F.from_json(F.col("value").cast("string"), schema).alias("books"))
                .select("books.*")
            .writeStream
                .option("checkpointLocation", f"{bookstore.checkpoint_path}/books_silver")
                .foreachBatch(type2_upsert)
                .trigger(availableNow=True)
                .start()
    )
    query.awaitTermination()

In [0]:
process_books()

In [0]:
books_df = (spark.read.table("books_silver").orderBy("book_id","effective_date"))
display(books_df)

In [0]:
bookstore.load_books_updates()

In [0]:
process_books()

In [0]:
%sql
CREATE OR REPLACE TABLE current_books
AS SELECT book_id, title, author, price
   FROM books_silver
   WHERE current IS TRUE

In [0]:
%sql 
select * from current_books